# Sign Language Detection System

This notebook provides an end-to-end sign language recognition workflow using:

**Webcam → MediaPipe Holistic → 1,662 landmark features per frame → 30-frame sequence → LSTM → sign prediction**

## System Overview

The project supports dataset preparation, model training and evaluation, real-time recognition, and user-managed action classes through a graphical interface.

### Key capabilities
- Consistent dataset loading and class organization
- Stratified training, validation, and test sets
- Validation-based early stopping and learning-rate adjustment
- Class weighting for imbalanced datasets
- Reproducible training through fixed random seeds
- Accuracy, precision, recall, Macro F1, and confusion-matrix evaluation
- Confidence-based real-time prediction with temporal smoothing
- Model and label-map persistence
- GUI-based action management, data collection, retraining, and recognition
- Optional support for a `none` / idle class

> **Dataset compatibility:** The default representation contains 1,662 values per frame and is compatible with the existing `MP_Data` structure. A different landmark representation requires a corresponding dataset and model to be trained from that representation.


In [ ]:
# for jupyter
%pip install -U numpy pandas tensorflow scikit-learn opencv-python mediapipe matplotlib seaborn

In [ ]:
#for collab
#!pip install -U numpy pandas tensorflow scikit-learn opencv-python mediapipe matplotlib seaborn

In [ ]:
import os
import json
import random
import time
import shutil
import threading
import re
from collections import Counter, deque

import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mediapipe as mp
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_PATH = "MP_Data"
MODEL_PATH = "action_model.keras"
LABELS_PATH = "labels.json"

SEQUENCE_LENGTH = 30
FEATURES_PER_FRAME = 1662

TEST_SIZE = 0.20
VAL_SIZE = 0.20

CONFIDENCE_THRESHOLD = 0.70
SMOOTHING_WINDOW = 8
STABLE_VOTES = 6

print("TensorFlow:", tf.__version__)
print("MediaPipe:", mp.__version__)

In [ ]:
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

def mediapipe_detection(image, model):
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_rgb.flags.writeable = False
    results = model.process(image_rgb)
    image_rgb.flags.writeable = True
    return cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR), results

def draw_styled_landmarks(image, results):
    mp_drawing.draw_landmarks(
        image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION,
        mp_drawing.DrawingSpec(color=(80, 110, 10), thickness=1, circle_radius=1),
        mp_drawing.DrawingSpec(color=(80, 255, 121), thickness=1, circle_radius=1)
    )
    mp_drawing.draw_landmarks(
        image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
        mp_drawing.DrawingSpec(color=(80, 22, 10), thickness=2, circle_radius=4),
        mp_drawing.DrawingSpec(color=(80, 44, 121), thickness=2, circle_radius=2)
    )
    mp_drawing.draw_landmarks(
        image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
        mp_drawing.DrawingSpec(color=(121, 22, 76), thickness=2, circle_radius=4),
        mp_drawing.DrawingSpec(color=(121, 44, 250), thickness=2, circle_radius=2)
    )
    mp_drawing.draw_landmarks(
        image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
        mp_drawing.DrawingSpec(color=(245, 117, 66), thickness=2, circle_radius=4),
        mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=2)
    )

def extract_keypoints(results):
    pose = (
        np.array([[r.x, r.y, r.z, r.visibility]
                  for r in results.pose_landmarks.landmark], dtype=np.float32).flatten()
        if results.pose_landmarks else np.zeros(33 * 4, dtype=np.float32)
    )
    face = (
        np.array([[r.x, r.y, r.z]
                  for r in results.face_landmarks.landmark], dtype=np.float32).flatten()
        if results.face_landmarks else np.zeros(468 * 3, dtype=np.float32)
    )
    left_hand = (
        np.array([[r.x, r.y, r.z]
                  for r in results.left_hand_landmarks.landmark], dtype=np.float32).flatten()
        if results.left_hand_landmarks else np.zeros(21 * 3, dtype=np.float32)
    )
    right_hand = (
        np.array([[r.x, r.y, r.z]
                  for r in results.right_hand_landmarks.landmark], dtype=np.float32).flatten()
        if results.right_hand_landmarks else np.zeros(21 * 3, dtype=np.float32)
    )
    return np.concatenate([pose, face, left_hand, right_hand]).astype(np.float32)

assert extract_keypoints.__name__ == "extract_keypoints"
print("Expected feature count:", FEATURES_PER_FRAME)

## 1. Discover the dataset safely

The original notebook used `os.listdir()` directly. This version sorts the folders and ignores non-directory files so class IDs remain reproducible.

In [ ]:
if not os.path.isdir(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset folder '{DATA_PATH}' was not found. "
        "Run the data-collection section first or place MP_Data beside this notebook."
    )

actions = np.array(sorted(
    name for name in os.listdir(DATA_PATH)
    if os.path.isdir(os.path.join(DATA_PATH, name))
))

if len(actions) < 2:
    raise ValueError(f"Need at least 2 action folders; found: {actions.tolist()}")

label_map = {action: idx for idx, action in enumerate(actions)}

print("Actions:", actions)
print("Label map:", label_map)

## 2. Dataset integrity check

This catches incomplete sequences before training instead of failing later inside the model.

In [ ]:
def numeric_sequence_dirs(action_path):
    result = []
    for name in os.listdir(action_path):
        full = os.path.join(action_path, name)
        if os.path.isdir(full):
            try:
                result.append(int(name))
            except ValueError:
                pass
    return sorted(result)

def load_dataset(data_path, actions, sequence_length, expected_features):
    sequences, labels, sequence_names = [], [], []
    skipped = []

    for action in actions:
        action_path = os.path.join(data_path, action)

        for seq_id in numeric_sequence_dirs(action_path):
            seq_dir = os.path.join(action_path, str(seq_id))
            frame_files = []

            valid = True
            for frame_num in range(sequence_length):
                frame_path = os.path.join(seq_dir, f"{frame_num}.npy")
                if not os.path.isfile(frame_path):
                    valid = False
                    break
                frame_files.append(frame_path)

            if not valid:
                skipped.append((action, seq_id, "missing frame"))
                continue

            window = []
            for frame_path in frame_files:
                arr = np.load(frame_path)
                arr = np.asarray(arr, dtype=np.float32).reshape(-1)

                if arr.size != expected_features:
                    valid = False
                    skipped.append((action, seq_id, f"expected {expected_features}, got {arr.size}"))
                    break

                if not np.all(np.isfinite(arr)):
                    valid = False
                    skipped.append((action, seq_id, "NaN/Inf"))
                    break

                window.append(arr)

            if valid:
                sequences.append(np.stack(window))
                labels.append(label_map[action])
                sequence_names.append((action, seq_id))

    if not sequences:
        raise ValueError("No valid sequences were found.")

    return (
        np.asarray(sequences, dtype=np.float32),
        np.asarray(labels, dtype=np.int64),
        sequence_names,
        skipped,
    )

X, y, sequence_names, skipped = load_dataset(
    DATA_PATH, actions, SEQUENCE_LENGTH, FEATURES_PER_FRAME
)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Valid sequences:", len(X))
print("Skipped sequences:", len(skipped))

if skipped:
    print("First skipped examples:")
    for item in skipped[:10]:
        print(" ", item)

In [ ]:
# Class distribution
counts = Counter(y)
distribution = {actions[k]: counts.get(k, 0) for k in range(len(actions))}
print("Class distribution:", distribution)

plt.figure(figsize=(10, 4))
plt.bar(list(distribution.keys()), list(distribution.values()))
plt.title("Sequences per class")
plt.xlabel("Action")
plt.ylabel("Number of sequences")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 3. Train / Validation / Test Split

The dataset is divided into training, validation, and test subsets using stratified sampling so that each action is represented consistently across the splits.

- **Training set:** used to learn model parameters
- **Validation set:** used during training for model selection and learning-rate scheduling
- **Test set:** reserved for final performance evaluation

The test set remains separate from model training so that the reported evaluation reflects performance on previously unseen sequences.


In [ ]:
# First: hold out the final test set.
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y
)

# Second: split the remaining training data into train and validation.
val_fraction_of_remaining = VAL_SIZE / (1.0 - TEST_SIZE)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=val_fraction_of_remaining,
    random_state=SEED,
    stratify=y_train_full
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

print("\nTrain distribution:", Counter(y_train))
print("Validation distribution:", Counter(y_val))
print("Test distribution:", Counter(y_test))

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_ids = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=class_ids,
    y=y_train
)
class_weight = {int(cls): float(w) for cls, w in zip(class_ids, weights)}
print("Class weights:", class_weight)

## 4. Model

This model is deliberately smaller than the original 3-LSTM stack. With a relatively small dataset, a smaller network plus regularization is easier to train and less likely to memorize the training examples.

The `GaussianNoise` layer is only active during training and provides mild landmark augmentation.

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, GaussianNoise
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard

model = Sequential([
    Input(shape=(SEQUENCE_LENGTH, FEATURES_PER_FRAME)),

    GaussianNoise(0.003),

    LSTM(64, return_sequences=True, activation="tanh"),
    Dropout(0.30),

    LSTM(64, return_sequences=False, activation="tanh"),
    Dropout(0.30),

    Dense(64, activation="relu"),
    Dropout(0.30),

    Dense(len(actions), activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=20,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=7,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        MODEL_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    TensorBoard(log_dir="logs/sign_language")
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=300,
    batch_size=16,
    class_weight=class_weight,
    callbacks=callbacks,
    shuffle=True,
    verbose=1
)

In [ ]:
# Plot training history
fig = plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Validation")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

## 5. Final evaluation on the untouched test set

Do not use the test set for tuning. It is reserved for the final performance estimate.

In [ ]:
# Reload the best checkpoint saved during training.
model = tf.keras.models.load_model(MODEL_PATH)

test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
y_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Macro F1: {f1_score(y_test, y_pred, average='macro'):.4f}")

print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    labels=np.arange(len(actions)),
    target_names=actions,
    zero_division=0
))

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=np.arange(len(actions))
)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=actions,
    yticklabels=actions
)
plt.title("Confusion Matrix — Test Set")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
# Save a reproducible label map next to the model.
with open(LABELS_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            "actions": actions.tolist(),
            "sequence_length": SEQUENCE_LENGTH,
            "features_per_frame": FEATURES_PER_FRAME,
            "confidence_threshold": CONFIDENCE_THRESHOLD
        },
        f,
        indent=2
    )

print("Saved:", MODEL_PATH)
print("Saved:", LABELS_PATH)

## 6. Real-Time Webcam Recognition

The recognition pipeline collects a rolling sequence of webcam frames and converts each frame into the same 1,662-feature representation used during training.

Predictions are stabilized over a short history using majority voting and a confidence threshold. This reduces rapid changes between labels and helps the displayed result remain stable during continuous signing.

The recognition window displays the detected action, prediction confidence, and the accumulated sentence. Use **Q** to close the recognition window and **C** to clear the current sentence.


In [ ]:
def stable_prediction(prediction_history, probabilities, threshold=CONFIDENCE_THRESHOLD):
    if not prediction_history:
        return None, 0.0

    # Only accept a label when recent predictions agree strongly enough.
    counts = Counter(prediction_history)
    label, votes = counts.most_common(1)[0]

    vote_ratio = votes / len(prediction_history)
    confidence = float(probabilities[label])

    if vote_ratio >= (STABLE_VOTES / SMOOTHING_WINDOW) and confidence >= threshold:
        return int(label), confidence

    return None, confidence

In [ ]:
def run_realtime(model, actions, threshold=CONFIDENCE_THRESHOLD):
    sequence = deque(maxlen=SEQUENCE_LENGTH)
    prediction_history = deque(maxlen=SMOOTHING_WINDOW)
    sentence = []

    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        raise RuntimeError(
            "Could not open webcam. Check camera permissions or try another camera index."
        )

    # Reduce unnecessary resolution for smoother inference.
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 960)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 540)

    with mp_holistic.Holistic(
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holistic:

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                print("Could not read a frame from the webcam.")
                break

            image, results = mediapipe_detection(frame, holistic)
            draw_styled_landmarks(image, results)

            keypoints = extract_keypoints(results)
            sequence.append(keypoints)

            current_label = "Waiting..."
            current_conf = 0.0

            if len(sequence) == SEQUENCE_LENGTH:
                probabilities = model.predict(
                    np.expand_dims(np.asarray(sequence), axis=0),
                    verbose=0
                )[0]

                predicted_idx = int(np.argmax(probabilities))
                current_conf = float(probabilities[predicted_idx])

                # Add the prediction only when its confidence is reasonable.
                if current_conf >= threshold:
                    prediction_history.append(predicted_idx)

                stable_idx, stable_conf = stable_prediction(
                    prediction_history,
                    probabilities,
                    threshold=CONFIDENCE_THRESHOLD
                )

                if stable_idx is not None:
                    current_label = actions[stable_idx]

                    # Add only a new consecutive word.
                    if not sentence or sentence[-1] != current_label:
                        sentence.append(current_label)

                    # Keep display manageable.
                    sentence = sentence[-6:]
                else:
                    current_label = actions[predicted_idx]

            # Header
            cv2.rectangle(image, (0, 0), (960, 90), (30, 30, 30), -1)

            cv2.putText(
                image,
                f"Sign: {current_label}",
                (15, 35),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.9,
                (255, 255, 255),
                2,
                cv2.LINE_AA
            )

            cv2.putText(
                image,
                f"Confidence: {current_conf:.2f}",
                (15, 70),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                (255, 255, 255),
                2,
                cv2.LINE_AA
            )

            # Sentence area
            cv2.rectangle(image, (0, 450), (960, 540), (30, 30, 30), -1)
            cv2.putText(
                image,
                "Sentence: " + " ".join(sentence),
                (15, 500),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.75,
                (255, 255, 255),
                2,
                cv2.LINE_AA
            )

            cv2.imshow("Sign Language Detection", image)

            key = cv2.waitKey(1) & 0xFF

            if key == ord("q"):
                break
            elif key == ord("c"):
                sentence.clear()
                prediction_history.clear()

    cap.release()
    cv2.destroyAllWindows()

print("Press Q to quit. Press C to clear the sentence.")

In [ ]:
# Uncomment this cell when you are ready to run the webcam.
# run_realtime(model, actions)

## 7. Data Collection

Use the data collection utility when creating a new action or expanding an existing dataset. Each sequence contains **30 frames**, with **1,662 features extracted from each frame**.

### Recommended collection practice

For each action, aim for **80–100 or more sequences** when possible. Vary the following conditions to improve generalization:
- signer position
- distance from the camera
- lighting
- background
- signing speed
- hand position

For a stronger evaluation, include data from multiple people and reserve at least one signer for testing only.

### `none` / background class

A `none` or idle class can represent periods when no target sign is being performed. This allows the system to distinguish intentional signs from ordinary idle frames.

If this class is included, collect representative idle examples rather than creating an empty action folder.


In [ ]:
def safe_action_name(name):
    """Return a safe folder/class name or raise ValueError."""
    name = name.strip()
    if not name:
        raise ValueError("Action name cannot be empty.")
    if len(name) > 40:
        raise ValueError("Action name must be 40 characters or fewer.")
    if name in {".", ".."} or "/" in name or "\\" in name:
        raise ValueError("Action name cannot contain path separators.")
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9 _-]*", name):
        raise ValueError(
            "Use letters, numbers, spaces, hyphens or underscores only."
        )
    return name

def get_actions_from_disk():
    return np.array(sorted(
        name for name in os.listdir(DATA_PATH)
        if os.path.isdir(os.path.join(DATA_PATH, name))
    ))

def collect_action(action, sequences_per_action=100, sequence_length=30):
    """Record one new action using the same 1,662-feature format."""
    action = safe_action_name(action)
    os.makedirs(DATA_PATH, exist_ok=True)

    action_path = os.path.join(DATA_PATH, action)
    os.makedirs(action_path, exist_ok=True)

    existing = numeric_sequence_dirs(action_path)
    start_sequence = max(existing) + 1 if existing else 0

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError(
            "Could not open webcam. Check camera permissions."
        )

    try:
        with mp_holistic.Holistic(
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        ) as holistic:

            for seq_offset in range(int(sequences_per_action)):
                sequence_id = start_sequence + seq_offset
                sequence_path = os.path.join(
                    action_path, str(sequence_id)
                )
                os.makedirs(sequence_path, exist_ok=True)

                for frame_num in range(sequence_length):
                    ret, frame = cap.read()
                    if not ret:
                        raise RuntimeError("Could not read webcam frame.")

                    image, results = mediapipe_detection(frame, holistic)
                    draw_styled_landmarks(image, results)

                    if frame_num == 0:
                        cv2.putText(
                            image,
                            "GET READY",
                            (120, 180),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            1.2,
                            (0, 255, 0),
                            3,
                            cv2.LINE_AA
                        )
                        cv2.putText(
                            image,
                            f"{action} | sequence {sequence_id}",
                            (15, 35),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.65,
                            (255, 255, 255),
                            2,
                            cv2.LINE_AA
                        )
                        cv2.imshow("Data Collection", image)
                        if cv2.waitKey(700) & 0xFF == ord("q"):
                            return

                    keypoints = extract_keypoints(results)
                    np.save(
                        os.path.join(sequence_path, f"{frame_num}.npy"),
                        keypoints
                    )

                    cv2.putText(
                        image,
                        f"{action} | {seq_offset+1}/{sequences_per_action} "
                        f"| frame {frame_num+1}/{sequence_length}",
                        (15, 35),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.55,
                        (255, 255, 255),
                        2,
                        cv2.LINE_AA
                    )
                    cv2.imshow("Data Collection", image)

                    if cv2.waitKey(1) & 0xFF == ord("q"):
                        return
    finally:
        cap.release()
        cv2.destroyAllWindows()

    print(
        f"Finished recording '{action}'. "
        f"Total sequences now: {len(numeric_sequence_dirs(action_path))}"
    )

## 8. User Interface — Action Management and Model Training

The graphical interface provides a simple workflow for managing the action classes without requiring changes to the Python source code.

### User workflow

1. Enter the name of a new action.
2. Select the number of sequences to record.
3. Choose **Add & Record** and perform the sign during data collection.
4. Select an existing action and use **Delete Action** when it is no longer required.
5. Use **Retrain Model** after adding or removing actions.
6. Use **Start Recognition** to run the trained model with the current action set.

### Important

Changes to the action folders change the classification problem. A new model should therefore be trained whenever actions are added or removed.


In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox

gui_state = {
    "model_ready": os.path.isfile(MODEL_PATH),
    "training": False,
}

def rebuild_dataset_and_model():
    """
    Reload current folders, perform the training/validation/test split, train a fresh model,
    evaluate it, and save the model + label map.
    """
    global X, y, X_train, X_val, X_test
    global y_train, y_val, y_test, actions, label_map, model

    actions = get_actions_from_disk()

    if len(actions) < 2:
        raise ValueError("At least two actions are required before training.")

    # Load all current action folders.
    X, y, sequence_names, skipped = load_dataset(
        DATA_PATH, actions, SEQUENCE_LENGTH, FEATURES_PER_FRAME
    )

    if len(X) < 10:
        raise ValueError("Not enough valid sequences to train a model.")

    per_class = Counter(y)
    too_small = [
        actions[idx] for idx in range(len(actions))
        if per_class.get(idx, 0) < 2
    ]
    if too_small:
        raise ValueError(
            "Every action needs at least 2 valid sequences. "
            f"Too few: {too_small}"
        )

    label_map = {action: idx for idx, action in enumerate(actions)}

    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=y
    )

    val_fraction_of_remaining = VAL_SIZE / (1.0 - TEST_SIZE)

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full,
        test_size=val_fraction_of_remaining,
        random_state=SEED,
        stratify=y_train_full
    )

    class_ids = np.unique(y_train)
    weights = compute_class_weight(
        class_weight="balanced",
        classes=class_ids,
        y=y_train
    )
    class_weight = {
        int(cls): float(weight)
        for cls, weight in zip(class_ids, weights)
    }

    fresh_model = Sequential([
        Input(shape=(SEQUENCE_LENGTH, FEATURES_PER_FRAME)),
        GaussianNoise(0.003),
        LSTM(64, return_sequences=True, activation="tanh"),
        Dropout(0.30),
        LSTM(64, return_sequences=False, activation="tanh"),
        Dropout(0.30),
        Dense(64, activation="relu"),
        Dropout(0.30),
        Dense(len(actions), activation="softmax")
    ])

    fresh_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    gui_callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=20,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=7,
            min_lr=1e-6,
            verbose=1
        ),
        ModelCheckpoint(
            MODEL_PATH,
            monitor="val_loss",
            save_best_only=True,
            verbose=1
        )
    ]

    history = fresh_model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=300,
        batch_size=16,
        class_weight=class_weight,
        callbacks=gui_callbacks,
        shuffle=True,
        verbose=1
    )

    # Load the best checkpoint rather than necessarily the final epoch.
    model = tf.keras.models.load_model(MODEL_PATH)

    test_loss, test_accuracy = model.evaluate(
        X_test, y_test, verbose=0
    )
    y_prob = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    macro_f1 = f1_score(y_test, y_pred, average="macro")

    with open(LABELS_PATH, "w", encoding="utf-8") as f:
        json.dump(
            {
                "actions": actions.tolist(),
                "sequence_length": SEQUENCE_LENGTH,
                "features_per_frame": FEATURES_PER_FRAME,
                "confidence_threshold": CONFIDENCE_THRESHOLD
            },
            f,
            indent=2
        )

    return {
        "actions": actions.copy(),
        "sequences": len(X),
        "test_accuracy": float(test_accuracy),
        "macro_f1": float(macro_f1),
        "skipped": len(skipped),
        "history": history
    }


def start_gui():
    root = tk.Tk()
    root.title("Sign Language Detection")
    root.geometry("720x620")
    root.resizable(False, False)

    title = ttk.Label(
        root,
        text="SIGN LANGUAGE DETECTION",
        font=("Arial", 20, "bold")
    )
    title.pack(pady=(18, 4))

    subtitle = ttk.Label(
        root,
        text="Add actions • Record data • Retrain • Recognize",
        font=("Arial", 10)
    )
    subtitle.pack(pady=(0, 15))

    status_var = tk.StringVar(value="Ready")
    action_var = tk.StringVar()
    samples_var = tk.StringVar(value="100")

    # ---------- Actions list ----------
    list_frame = ttk.LabelFrame(root, text="Current Actions")
    list_frame.pack(fill="x", padx=25, pady=8)

    listbox = tk.Listbox(
        list_frame,
        height=8,
        font=("Arial", 11),
        selectmode=tk.SINGLE
    )
    listbox.pack(side="left", fill="x", expand=True, padx=10, pady=10)

    scrollbar = ttk.Scrollbar(
        list_frame,
        orient="vertical",
        command=listbox.yview
    )
    scrollbar.pack(side="right", fill="y", pady=10)
    listbox.config(yscrollcommand=scrollbar.set)

    def refresh_actions():
        listbox.delete(0, tk.END)
        for action in get_actions_from_disk():
            listbox.insert(tk.END, action)

        status_var.set(
            f"{listbox.size()} action(s) found"
        )

    # ---------- Add action ----------
    add_frame = ttk.LabelFrame(root, text="Add New Action")
    add_frame.pack(fill="x", padx=25, pady=8)

    ttk.Label(add_frame, text="Action name:").grid(
        row=0, column=0, padx=10, pady=10, sticky="w"
    )

    action_entry = ttk.Entry(
        add_frame,
        textvariable=action_var,
        width=30
    )
    action_entry.grid(
        row=0, column=1, padx=10, pady=10, sticky="w"
    )

    ttk.Label(add_frame, text="Sequences:").grid(
        row=0, column=2, padx=10, pady=10, sticky="w"
    )

    samples_entry = ttk.Entry(
        add_frame,
        textvariable=samples_var,
        width=8
    )
    samples_entry.grid(
        row=0, column=3, padx=10, pady=10
    )

    def add_and_record():
        if gui_state["training"]:
            messagebox.showwarning(
                "Busy",
                "Please wait until training finishes."
            )
            return

        try:
            action = safe_action_name(action_var.get())

            try:
                count = int(samples_var.get())
            except ValueError:
                raise ValueError("Sequences must be a whole number.")

            if count < 2:
                raise ValueError("Use at least 2 sequences.")
            if count > 1000:
                raise ValueError("Maximum allowed is 1000 sequences.")

            existing = set(get_actions_from_disk().tolist())
            if action in existing:
                messagebox.showerror(
                    "Action exists",
                    f"'{action}' already exists. "
                    "Choose another name or delete the existing action first."
                )
                return

            status_var.set(
                f"Recording '{action}'... Press Q in the camera window to stop."
            )
            root.update_idletasks()

            collect_action(
                action,
                sequences_per_action=count,
                sequence_length=SEQUENCE_LENGTH
            )

            action_var.set("")
            refresh_actions()

            gui_state["model_ready"] = False
            status_var.set(
                f"'{action}' added. RETRAIN MODEL before recognition."
            )

            messagebox.showinfo(
                "Action added",
                f"Recorded '{action}'.\\n\\n"
                "Click 'Retrain Model' to include it in the classifier."
            )

        except Exception as exc:
            messagebox.showerror("Recording error", str(exc))
            status_var.set("Recording failed")

    ttk.Button(
        add_frame,
        text="ADD & RECORD",
        command=add_and_record
    ).grid(
        row=0, column=4, padx=12, pady=10
    )

    # ---------- Delete ----------
    def delete_selected():
        if gui_state["training"]:
            messagebox.showwarning(
                "Busy",
                "Please wait until training finishes."
            )
            return

        selected = listbox.curselection()
        if not selected:
            messagebox.showwarning(
                "Select action",
                "Select an action to delete first."
            )
            return

        action = listbox.get(selected[0])

        if len(get_actions_from_disk()) <= 2:
            messagebox.showerror(
                "Cannot delete",
                "Keep at least two actions so the classifier can be trained."
            )
            return

        confirm = messagebox.askyesno(
            "Delete action",
            f"Delete '{action}' and ALL of its recorded data?"
        )

        if not confirm:
            return

        try:
            shutil.rmtree(os.path.join(DATA_PATH, action))
            refresh_actions()

            gui_state["model_ready"] = False
            status_var.set(
                f"'{action}' deleted. RETRAIN MODEL before recognition."
            )

        except Exception as exc:
            messagebox.showerror("Delete error", str(exc))

    ttk.Button(
        list_frame,
        text="DELETE SELECTED",
        command=delete_selected
    ).pack(
        side="bottom",
        pady=(0, 10)
    )

    # ---------- Training ----------
    control_frame = ttk.LabelFrame(root, text="Model")
    control_frame.pack(fill="x", padx=25, pady=8)

    def set_status(message):
        status_var.set(message)
        root.update_idletasks()

    def retrain_worker():
        try:
            set_status(
                "Training in progress... You can watch the notebook output."
            )

            result = rebuild_dataset_and_model()

            gui_state["model_ready"] = True

            def finish():
                set_status(
                    f"Training complete | "
                    f"Accuracy: {result['test_accuracy']:.2%} | "
                    f"Macro F1: {result['macro_f1']:.2%}"
                )
                messagebox.showinfo(
                    "Training complete",
                    f"Model retrained successfully.\\n\\n"
                    f"Actions: {len(result['actions'])}\\n"
                    f"Sequences: {result['sequences']}\\n"
                    f"Test accuracy: {result['test_accuracy']:.2%}\\n"
                    f"Macro F1: {result['macro_f1']:.2%}"
                )
                gui_state["training"] = False

            root.after(0, finish)

        except Exception as exc:
            gui_state["training"] = False

            def fail():
                set_status("Training failed")
                messagebox.showerror("Training error", str(exc))

            root.after(0, fail)

    def retrain():
        if gui_state["training"]:
            return

        if len(get_actions_from_disk()) < 2:
            messagebox.showerror(
                "Not enough actions",
                "Add at least two actions before training."
            )
            return

        gui_state["training"] = True
        threading.Thread(
            target=retrain_worker,
            daemon=True
        ).start()

    ttk.Button(
        control_frame,
        text="RETRAIN MODEL",
        command=retrain
    ).pack(side="left", padx=12, pady=12)

    def start_recognition():
        if gui_state["training"]:
            messagebox.showwarning(
                "Busy",
                "Please wait until training finishes."
            )
            return

        if not os.path.isfile(MODEL_PATH):
            messagebox.showwarning(
                "No model",
                "Train a model before starting recognition."
            )
            return

        try:
            current_actions = get_actions_from_disk()
            if len(current_actions) < 2:
                raise ValueError("At least two actions are required.")

            # If the disk classes differ from the saved label map, force retraining.
            if os.path.isfile(LABELS_PATH):
                with open(LABELS_PATH, "r", encoding="utf-8") as f:
                    saved = json.load(f)
                saved_actions = saved.get("actions", [])
                if list(current_actions) != list(saved_actions):
                    messagebox.showwarning(
                        "Retrain required",
                        "The action set has changed since the last training. "
                        "Please retrain the model first."
                    )
                    return

            current_model = tf.keras.models.load_model(MODEL_PATH)
            run_realtime(current_model, current_actions)

        except Exception as exc:
            messagebox.showerror("Recognition error", str(exc))

    ttk.Button(
        control_frame,
        text="START RECOGNITION",
        command=start_recognition
    ).pack(side="left", padx=12, pady=12)

    ttk.Button(
        control_frame,
        text="REFRESH ACTIONS",
        command=refresh_actions
    ).pack(side="left", padx=12, pady=12)

    # ---------- Status ----------
    status_frame = ttk.LabelFrame(root, text="Status")
    status_frame.pack(fill="x", padx=25, pady=8)

    ttk.Label(
        status_frame,
        textvariable=status_var,
        font=("Arial", 10, "bold"),
        wraplength=630
    ).pack(padx=15, pady=14)

    ttk.Label(
        root,
        text=(
            "Tip: collect 80–100+ sequences per action. "
            "Use Q to stop camera recording and Q/C during recognition."
        ),
        wraplength=650,
        justify="center"
    ).pack(pady=8)

    refresh_actions()
    root.mainloop()

print("GUI ready. Run: start_gui()")

## 8. Optional future upgrade: pose + hands only

Your current files contain face landmarks, so the default notebook retains them.

For a future experiment, remove the face landmarks and use:
- pose: 132 values
- left hand: 63 values
- right hand: 63 values

Total: **258 values/frame**.

That can make the model smaller and faster, but it requires recollecting the dataset because the saved `.npy` files must have 258 values/frame.

In [ ]:
def extract_keypoints_pose_hands_only(results):
    pose = (
        np.array([[r.x, r.y, r.z, r.visibility]
                  for r in results.pose_landmarks.landmark], dtype=np.float32).flatten()
        if results.pose_landmarks else np.zeros(33 * 4, dtype=np.float32)
    )
    left_hand = (
        np.array([[r.x, r.y, r.z]
                  for r in results.left_hand_landmarks.landmark], dtype=np.float32).flatten()
        if results.left_hand_landmarks else np.zeros(21 * 3, dtype=np.float32)
    )
    right_hand = (
        np.array([[r.x, r.y, r.z]
                  for r in results.right_hand_landmarks.landmark], dtype=np.float32).flatten()
        if results.right_hand_landmarks else np.zeros(21 * 3, dtype=np.float32)
    )
    return np.concatenate([pose, left_hand, right_hand]).astype(np.float32)

print("Future pose+hands feature count:", 33*4 + 21*3 + 21*3)

## 10. Project Checklist

Before a final demonstration or evaluation, verify the following:

- [ ] Collect sufficient sequences for each action.
- [ ] Include a `none` / idle class when appropriate.
- [ ] Test with at least one signer not represented in the training data.
- [ ] Keep the test set separate until final evaluation.
- [ ] Report accuracy, Macro F1, precision, recall, and the confusion matrix.
- [ ] Evaluate performance under different lighting, backgrounds, distances, and signing speeds.
- [ ] Compare the full landmark representation with a pose-and-hands-only representation as an optional experiment.
- [ ] Keep `action_model.keras` and `labels.json` together when moving or deploying the trained model.
